# Exercise 4: Absent languages

*Corpora at Scale · version 0.1.0*

> Choose one language that is absent from HPLT v3.0. Find three web sources for it, estimate how much text they hold, and run GlotLID on a sample. Under which labels would this text end up in a web corpus?

**Setup.** `pip install "turkicnlp==0.5.0" pyhfst fasttext-wheel huggingface_hub requests`. Open the notebook from the `exercises/` folder: the first code cell imports the chapter's audit functions from `../code/remaining/corpus.py`. GlotLID (`model_v3.bin`, about 1.6 GB) is downloaded from Hugging Face once; set `GLOTLID_PATH` to use a local copy. The Apertium analysers come with TurkicNLP and are read with the pure-Python `pyhfst`. The text comes from the X-WMT test sets on GitHub, so internet access is needed.

Sakha and Karakalpak are both absent from HPLT v3.0 (see the chapter's HPLT v3.0 table) and both have X-WMT test sets, which serve here as the sample. The code runs unrestricted GlotLID and then restricts it to an HPLT-like label set: the eleven Turkic subsets of HPLT v3.0 plus Russian and English. HPLT v3.0 itself uses OpenLID-v2, not GlotLID, so the restricted result only approximates which existing subset the text would join. Finding web sources and estimating their size is manual work; the last cell lists what to record.

In [ ]:
import os, re, sys, unicodedata
from collections import Counter
import requests

sys.path.insert(0, os.path.abspath("../code/remaining"))      # the book's corpus.py
from corpus import audit, script_shares, fst_coverage

BASE = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test"

def xwmt_side(pair, lang):
    """Non-empty lines of one side of an X-WMT test pair, e.g. xwmt_side("en-ky", "ky")."""
    r = requests.get(f"{BASE}/{pair}/{pair}.{lang}", timeout=30)
    r.raise_for_status()
    return [line.strip() for line in r.text.splitlines() if line.strip()]

def glotlid_model():
    """Unrestricted GlotLID v3 (fastText). Uses GLOTLID_PATH if set, otherwise downloads the model."""
    import fasttext
    path = os.environ.get("GLOTLID_PATH")
    if not path:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
    return fasttext.load_model(path)

def lid_labels(model, text, k=1):
    """Labels, best first. A list input avoids a NumPy 2 incompatibility in fasttext's single-string path."""
    return model.predict([text.replace("\n", " ")], k=k)[0][0]

def load_analyser(lang, script="Cyrl"):
    """Apertium analyser from TurkicNLP, read with pyhfst. lookup(word) -> [(reading, weight), ...]."""
    import pyhfst, turkicnlp
    from turkicnlp.resources.registry import ModelRegistry
    turkicnlp.download(lang, processors=["morph"])
    path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.automorf.hfst"
    fst = pyhfst.HfstInputStream(str(path)).read()
    def lookup(word):
        readings = fst.lookup(word) or fst.lookup(word.lower())
        return [(reading, float(weight)) for reading, weight in readings]
    return lookup

WORD = re.compile(r"[^\W\d_]+")
def words(text):
    return WORD.findall(unicodedata.normalize("NFC", text))

In [ ]:
model = glotlid_model()
labels = model.get_labels()
HPLT_TURKIC = ["tur_Latn", "azj_Latn", "azb_Arab", "tuk_Latn", "kaz_Cyrl", "kir_Cyrl", "tat_Cyrl",
               "bak_Cyrl", "crh_Latn", "uzn_Latn", "uig_Arab"]          # the HPLT v3.0 subsets (tbl-hplt-v3)
ALLOWED = {f"__label__{c}" for c in HPLT_TURKIC + ["rus_Cyrl", "eng_Latn"]}

def ranked(text):
    return lid_labels(model, text, k=len(labels))                     # all labels, best first

for pair, lang, gold in [("en-sah", "sah", "__label__sah_Cyrl"), ("en-kaa", "kaa", "__label__kaa_Cyrl")]:
    lines = xwmt_side(pair, lang)
    unrestricted = Counter(ranked(s)[0] for s in lines)
    restricted = Counter(next(l for l in ranked(s) if l in ALLOWED) for s in lines)
    print(f"{lang}: {len(lines)} sentences; gold label correct for {unrestricted[gold]}")
    print("   unrestricted top labels:", unrestricted.most_common(4))
    print("   restricted to HPLT-like labels:", restricted.most_common(4))

**Sources.** For each candidate web source (for example a national newspaper, a regional broadcaster, a government portal, Wikipedia), record the URL, the publisher, the date checked, the licence or terms, and how you estimated the size (number of articles in the archive × a sample of article lengths in words). An estimate without its method is not usable.

## Worked solution

Output on 18 October 2026, UTC (GlotLID v3; X-WMT test sentences):

```
sah: 300 sentences; gold label correct for 300
   unrestricted top labels: [('__label__sah_Cyrl', 300)]
   restricted to HPLT-like labels: [('__label__kir_Cyrl', 206), ('__label__tat_Cyrl', 76), ('__label__rus_Cyrl', 11), ('__label__uzn_Latn', 6)]
kaa: 300 sentences; gold label correct for 277
   unrestricted top labels: [('__label__kaa_Cyrl', 277), ('__label__kaz_Cyrl', 13), ('__label__uzn_Cyrl', 5), ('__label__kir_Cyrl', 2)]
   restricted to HPLT-like labels: [('__label__kaz_Cyrl', 261), ('__label__kir_Cyrl', 37), ('__label__bak_Cyrl', 2)]
```

Unrestricted GlotLID labels all 300 Sakha sentences correctly and 277 of the 300 Karakalpak ones; the remaining Karakalpak sentences go to Kazakh, Cyrillic Uzbek or Kyrgyz. Restricted to labels that exist as HPLT v3.0 subsets, Sakha text goes mostly to Kyrgyz (206) and Tatar (76), some to Russian (11), and Karakalpak almost entirely to Kazakh (261) and Kyrgyz (37). This is an approximation: HPLT uses OpenLID-v2, not GlotLID, and its own filters (for example a minimum language-identification probability) may discard such text rather than keep it under a wrong label. It does show where to look: Sakha or Karakalpak text that reached HPLT at all would most likely sit in the Kyrgyz, Tatar or Kazakh subsets. The Karakalpak test set is in Cyrillic, while current official Karakalpak uses the Latin script, so a Latin-script sample may behave differently.

The source search and size estimates are not part of this worked solution: they need live web access and have to be dated.

## Your turn — extension

1. Choose Kumyk, Nogai or Khakas, which have no X-WMT test set. Collect 100 sentences from a source whose language you can verify, and run the same cell.
2. Repeat with the label set of a different corpus, for example the Turkic languages of CC-100 or mC4.
3. For your three web sources, report the size estimate with its method and the date.